In [ ]:
%load_ext autoreload
%autoreload 2

## Importing

In [ ]:
import pandas as pd
from scripts.data_utils import cut_obs_df
import matplotlib.pyplot as plt

import matplotlib.cm as cm
import numpy as np

from scipy.spatial.distance import euclidean, cityblock, jensenshannon
from scipy.stats import wasserstein_distance

import seaborn as sns

import osmnx as ox

## Helper functions

In [ ]:

def plot_weekday_vs_weekend_profile(
    df,
    aggr_method="mean",
    normalize=False,
    save_path = None
):

    df_weekday = df[df.index.weekday < 5]
    df_weekend = df[df.index.weekday >= 5]

    def compute_profile(data):
        hourly_totals = data.resample("h").sum()
        grouped = hourly_totals.groupby(hourly_totals.index.hour).mean()

        if aggr_method == "mean":
            mean_profile = grouped.mean(axis=1)
        elif aggr_method == "sum":
            mean_profile = grouped.sum(axis=1)
        else:
            raise ValueError("aggr_method must be 'mean' or 'sum'")

        std_profile = grouped.std(axis=1)

        mean_profile = mean_profile.reindex(range(24), fill_value=0)
        std_profile = std_profile.reindex(range(24), fill_value=0)

        if normalize:
            mean_profile = mean_profile / mean_profile.sum()
            std_profile = std_profile / mean_profile.sum()

        return mean_profile, std_profile

    prof_weekday, std_weekday = compute_profile(df_weekday)
    prof_weekend, std_weekend = compute_profile(df_weekend)

    plt.figure(figsize=(10,5))

    # Weekdays
    plt.plot(prof_weekday.index, prof_weekday.values, '-o',
             label="Weekdays", linewidth=2, markersize=5)
    # plt.fill_between(prof_weekday.index,
    #                  prof_weekday - std_weekday,
    #                  prof_weekday + std_weekday,
    #                  alpha=0.2)

    # Weekend
    plt.plot(prof_weekend.index, prof_weekend.values, '-o',
             label="Weekend", linewidth=2, markersize=5)
    # plt.fill_between(prof_weekend.index,
    #                  prof_weekend - std_weekend,
    #                  prof_weekend + std_weekend,
    #                  alpha=0.2)

    # plt.title("Hourly Traffic Profile: Weekdays vs Weekend (All sensors)",
    #           fontsize=16, fontweight='bold')
    plt.xlabel("Hour of Day")
    plt.ylabel("Average Traffic Volume" if not normalize else "Share of Daily Traffic")

    hours = range(24)
    plt.xticks(
        hours,
        [f"{h:02d}:00" for h in hours],
        rotation=45,
        ha="right",
        fontsize=9
    )
    plt.grid(True, linestyle="--", alpha=0.3, axis='y')
    plt.legend()
    plt.tight_layout()
    
    if save_path is not None:
        plt.savefig(save_path, format="pdf", bbox_inches="tight")
        
    plt.show()

    return prof_weekday, prof_weekend

In [ ]:

def plot_all_sensors_by_hour(df: pd.DataFrame, figsize=(10, 5), include_legend = False) -> plt.Figure:

    df_hourly = df.resample("h").sum()
    df_by_hour = df_hourly.groupby(df_hourly.index.hour).mean()

    hours = df_by_hour.index.values
    sensors = df_by_hour.columns.tolist()
    n = len(sensors)

    cmap = cm.get_cmap("turbo", n)
    colors = [cmap(i) for i in range(n)]

    fig, ax = plt.subplots(figsize=figsize)

    for i, sensor in enumerate(sensors):
        values = df_by_hour[sensor].values
        ax.plot(
            hours, values,
            color=colors[i],
            linewidth=1.2,
            alpha=0.75,
            label=sensor,
        )

    overall_mean = df_by_hour.mean(axis=1).values
    ax.plot(
        hours, overall_mean,
        color="white", linewidth=3.5, zorder=5
    )
    ax.plot(
        hours, overall_mean,
        color="black", linewidth=2.0, zorder=6,
        linestyle="--", label="Fleet mean"
    )

    ax.set_xticks(hours)
    ax.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45, ha="right", fontsize=9)
    # ax.set_xlabel("Hour of day", fontsize=11)
    # ax.set_ylabel("Avg vehicles / hour", fontsize=11)
    # ax.set_title(
    #     "Total hourly traffic volume per sensors",
    #     fontsize=13, pad=12
    # )
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5)
    ax.set_xlim(0, 23)

    if include_legend:
        legend = ax.legend(
            loc="upper left",
            bbox_to_anchor=(1.01, 1),
            borderaxespad=0,
            fontsize=6.5,
            ncol=2,              
            framealpha=0.9,
            title="Sensors",
            title_fontsize=8,
        )

    plt.tight_layout()
    # plt.savefig("traffic_all_curves.png", dpi=150, bbox_inches="tight")
    return fig

In [ ]:

def compute_hourly_profile(obs_df, aggr_method="mean", normalize=False):

    hourly_totals = obs_df.resample("h").sum().mean(axis=1)
    grouped = hourly_totals.groupby(hourly_totals.index.hour)

    if aggr_method == "mean":
        mean_profile = grouped.mean()
    elif aggr_method == "sum":
        mean_profile = grouped.sum()
    else:
        raise ValueError("aggr_method must be 'mean' or 'sum'")

    std_profile = grouped.std()

    mean_profile = mean_profile.reindex(range(24), fill_value=0)
    std_profile = std_profile.reindex(range(24), fill_value=0)

    if normalize:
        mean_profile = mean_profile / mean_profile.sum()
        std_profile = std_profile / mean_profile.sum()

    return mean_profile, std_profile
    # return mean_profile


def plot_hourly_comparison(df_before, df_during, df_after,
                           aggr_method="mean",
                           normalize=False,
                           save_path=None,
                           title = "Hourly Traffic Profile Comparison"
                           ):

    prof_before, std_before = compute_hourly_profile(df_before, aggr_method, normalize)
    prof_during, std_during = compute_hourly_profile(df_during, aggr_method, normalize)
    prof_after, std_after   = compute_hourly_profile(df_after,  aggr_method, normalize)

    plt.figure(figsize=(10,5))

    plt.plot(prof_before.index, prof_before.values,
             '-o', label="Before Closure")
    # plt.fill_between(prof_before.index,
    #                  prof_before - std_before,
    #                  prof_before + std_before,
    #                  alpha=0.2)

    plt.plot(prof_during.index, prof_during.values,
             '-o', label="During Closure")
    # plt.fill_between(prof_during.index,
    #                  prof_during - std_during,
    #                  prof_during + std_during,
    #                  alpha=0.2)

    plt.plot(prof_after.index, prof_after.values,
             '-o', label="After Closure")
    # plt.fill_between(prof_after.index,
    #                  prof_after - std_after,
    #                  prof_after + std_after,
    #                  alpha=0.2)

    # plt.title(title,
    #           fontsize=16, fontweight='bold')
    plt.xlabel("Hour of Day")
    plt.ylabel("Average Traffic Volume" if not normalize else "Share of Daily Traffic")

    hours = range(24)
    plt.xticks(
        hours,
        [f"{h:02d}:00" for h in hours],
        rotation=45,
        ha="right",
        fontsize=9
    )
    plt.grid(True, alpha=0.3, linestyle='--', axis='y')
    plt.legend()
    plt.tight_layout()
    
    if save_path is not None:
        plt.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.show()

    return prof_before, prof_during, prof_after

def compare_distributions(p1, p2):
    """
    p1, p2 are 24-length pandas Series
    Returns dictionary of distance metrics
    """
    
    # Ensure numpy arrays
    a = p1.values
    b = p2.values
    
    # Normalize for divergence metrics
    a_norm = a / a.sum()
    b_norm = b / b.sum()

    metrics = {
        "Euclidean Distance": euclidean(a, b),
        "Manhattan Distance": cityblock(a, b),
        "Wasserstein Distance": wasserstein_distance(range(24), range(24),
                                                     u_weights=a_norm,
                                                     v_weights=b_norm),
        "Jensen-Shannon Distance": jensenshannon(a_norm, b_norm)
    }
    
    return metrics

In [ ]:
def plot_hourly_zero_frequency(df, save_path=None):
    """
    df: pandas DataFrame
        - index must be datetime
        - columns = sensors
        - values = measurements
    """

    # Ensure datetime index
    df = df.copy()
    df.index = pd.to_datetime(df.index)

    # Create binary dataframe: 1 if zero, 0 otherwise
    zero_df = (df == 0).astype(int)

    # Group by hour and compute zero frequency per sensor
    hourly_zero_freq = zero_df.groupby(df.index.hour).sum()

    # Melt for seaborn
    melted = hourly_zero_freq.reset_index().melt(
        id_vars='index', var_name='sensor', value_name='zero_freq'
    )
    melted.rename(columns={'index': 'hour'}, inplace=True)

    # Plot
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=melted, x='hour', y='zero_freq', showfliers=False)

    # # Compute mean and std per hour
    # stats = melted.groupby('hour')['zero_freq'].agg(['mean', 'std']).reset_index()

    # # Overlay mean
    # plt.plot(stats['hour'], stats['mean'], color='red', marker='o', label='Mean')

    # # Overlay std as error bars
    # plt.errorbar(
    #     stats['hour'],
    #     stats['mean'],
    #     yerr=stats['std'],
    #     fmt='none',
    #     ecolor='red',
    #     capsize=3,
    #     label='Std'
    # )

    # plt.title('Hourly Zero Frequency Distribution')
    plt.xlabel('Hour of Day')
    plt.ylabel('Zero Frequency')
    
    hours = range(24)
    plt.xticks(
        hours,
        [f"{h:02d}:00" for h in hours],
        rotation=45,
        ha="right",
        fontsize=9
    )
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.3, axis='y')

    if save_path is not None:
        plt.savefig(save_path, format="pdf", bbox_inches="tight")
        
    plt.show()

## Load data

In [ ]:
from scripts.data_utils import merge_data_by_junc
import geopandas as gpd
from scripts.data_utils import from_df_to_geopandas
import pandas as pd

traffic_cams_df_with_dirs = gpd.read_file("./data/prod/pre-process/traffic_cams/gpd/traffic_cam_metadata.shp")
aggregated_timeseries = pd.read_csv("./data/prod/pre-process/traffic_cams/pd_time_series.csv", index_col=0, parse_dates=True)


In [ ]:
aggregated_timeseries.index = pd.to_datetime(aggregated_timeseries.index, utc=True).tz_convert("Europe/Rome")

In [ ]:
traffic_cams_df_with_dirs.crs

## Weekdays vs. Weekends

In [ ]:
prof_weekday, prof_weekend = plot_weekday_vs_weekend_profile(
    aggregated_timeseries,
    aggr_method="mean",
    normalize=False,
    save_path="./outputs/new_figs/weekday_vs_weekend_profile.pdf"
)

In [ ]:
weekdays = aggregated_timeseries[aggregated_timeseries.index.weekday < 5]
weekends = aggregated_timeseries[aggregated_timeseries.index.weekday >= 5]

fig = plot_all_sensors_by_hour(weekdays)
plt.show()

In [ ]:
 # Select time series of sensors whose name starts with SITO
aggregated_timeseries_sito = aggregated_timeseries[aggregated_timeseries.columns[aggregated_timeseries.columns.str.startswith("SITO")]]

In [ ]:
fig = plot_all_sensors_by_hour(aggregated_timeseries_sito, include_legend=True)

In [ ]:
fig = plot_all_sensors_by_hour(weekends)
plt.show()

## School closure

In [ ]:
school_sensors_north = [
    "SITO11-ADABASSANO", 
    "LT_2_009B_LT CAMERINI VS ROTATORIA ANNIBALE DA BASSANO", 
    "LT_2_009A_LT CAMERINI VS ROTATORIA CAVALCAVIA CAMERINI", 
    "LT_2_010B_LT VIANELLO-BUONARROTI VS ROTONDA BUONARROTI", 
    "LT_2_010A_LT VIANELLO-GIUCCIARDINI VS CAVALCAVIA CAMERINI", 
    "LT_2_011B_VIA RENI/VIA DUPRE VS CAVALCAVIA BORGOMAGNO/CENTRO STORICO", 
    "LT_2_011A_VIA RENI/VIA DUPRE VS TANGENZALE/VIGODARZERE"
]

school_sensors_ts = aggregated_timeseries[school_sensors_north]

data_9_11_feb_schools_north = cut_obs_df(school_sensors_ts, 1770591600000, 1770764400000)
data_16_18_feb_schools_north = cut_obs_df(school_sensors_ts, 1771196400000, 1771369200000)
data_23_25_feb_schools_north = cut_obs_df(school_sensors_ts, 1771801200000, 1771974000000)

In [ ]:
prof_before, prof_during, prof_after = plot_hourly_comparison(
    data_9_11_feb_schools_north, 
    data_16_18_feb_schools_north, 
    data_23_25_feb_schools_north, 
    aggr_method="mean", 
    normalize=False,
    title = "Hourly Traffic Profile Comparison (Sensors closed to schools)",
    save_path = "./outputs/new_figs/school_closure_comparison_schools_north.pdf"
)

In [ ]:
data_9_11_feb = cut_obs_df(aggregated_timeseries, 1770591600000, 1770764400000)
data_16_18_feb = cut_obs_df(aggregated_timeseries, 1771196400000, 1771369200000)
data_23_25_feb = cut_obs_df(aggregated_timeseries, 1771801200000, 1771974000000)

In [ ]:
prof_before, prof_during, prof_after = plot_hourly_comparison(
    data_9_11_feb, 
    data_16_18_feb, 
    data_23_25_feb, 
    aggr_method="mean", 
    normalize=False,
    save_path = "./outputs/new_figs/school_closure_comparison_all.pdf",
    title = "Hourly Traffic Profile Comparison (All sensors)"
)

## Zero values distr

In [ ]:
plot_hourly_zero_frequency(aggregated_timeseries, save_path="./outputs/new_figs/hourly_zero_frequency.pdf")

## Plots

In [ ]:
# import pydeck as pdk
# import pandas as pd

# # Sensors as nodes
# sensor_layer = pdk.Layer(
#     "ScatterplotLayer",
#     data=traffic_cams_df_with_dirs,
#     get_position=["LON", "LAT"],
#     get_radius=50,
#     get_fill_color=[255, 100, 0, 200],
# )
# view = pdk.ViewState(latitude=45.40622305006962, longitude=11.876285735508933, zoom=12, pitch=45)
# pdk.Deck(layers=[sensor_layer], initial_view_state=view).to_html("map.html")


In [ ]:
zone_file_path = "./data/prod/pre-process/context/SIT_SEZIONI_2021/SIT_SEZIONI_2021.shp"
pop_file_path = "./data/prod/pre-process/context/SIT_SEZIONI_2021/residenti_x_sezioneISTAT-2021.csv"

zones = gpd.read_file(zone_file_path)
residents = pd.read_csv(pop_file_path, delimiter=";")

zones_filtered = zones[zones['SEZ21'].isin(residents['Sezioni 2021 attribuite'])].copy()
mapping = residents.set_index("Sezioni 2021 attribuite")["Somma - Residenti"]
zones_filtered["POP21"] = zones_filtered["SEZ21"].map(mapping).fillna(zones_filtered["POP21"])

census_gdf = zones_filtered

df_pois = pd.read_csv("./data/prod/pre-process/context/filtered_grouped_poi.csv", index_col=0)
gdf_pois = gpd.GeoDataFrame(
    df_pois.drop(columns=["id"]),
    geometry=gpd.points_from_xy(df_pois.longitude, df_pois.latitude),
    crs="EPSG:4326"
)

In [ ]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import box
import pandas as pd

# --- Configuration ---
TARGET_CRS = "EPSG:32632"  # UTM 32N (Metric)
WGS84 = "EPSG:4326"        # Lat/Lon (Standard for OSM)

ROAD_STYLES = {
    "motorway":    ("#e05c5c", 1.4),
    "trunk":       ("#e08c5c", 1.1),
    "primary":     ("#888888", 0.8),
    "secondary":   ("#aaaaaa", 0.6),
    "tertiary":    ("#cccccc", 0.4),
    "residential": ("#dddddd", 0.3),
}

def get_plot_extent(gdfs, padding=500):
    """Calculates a combined bounding box in the target metric CRS."""
    valid_gdfs = [gdf for gdf in gdfs if gdf is not None and not gdf.empty]
    # Ensure all are in the same CRS for the calculation
    geoms = [gdf.to_crs(TARGET_CRS).geometry for gdf in valid_gdfs]
    combined_geoms = pd.concat(geoms)
    minx, miny, maxx, maxy = combined_geoms.total_bounds
    return [minx - padding, maxx + padding, miny - padding, maxy + padding]

def get_infrastructure_by_extent():
    G = ox.load_graphml("./data/prod/pre-process/road_network/osmnx.graphml")
    
    nodes, edges = ox.graph_to_gdfs(G)
    return nodes.to_crs(TARGET_CRS), edges.to_crs(TARGET_CRS)

def plot_base_infrastructure(ax, edges, extent):
    """Plots the road network within the extent."""
    for road_type, (color, lw) in ROAD_STYLES.items():
        subset = edges[edges["highway"].apply(
            lambda x: road_type in x if isinstance(x, list) else x == road_type
        )]
        if not subset.empty:
            subset.plot(ax=ax, color=color, linewidth=lw, zorder=1)
    
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_axis_off()

def generate_census_map(census_gdf, edges, extent, filename):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)
    
    census_gdf["POP_DENS"] = census_gdf["POP21"]
    census_gdf.to_crs(TARGET_CRS).plot(
        column="POP_DENS", ax=ax, cmap="YlOrRd",
        legend=True, legend_kwds={"label": "Zone Population", "shrink": 0.4}, zorder=2
    )
    
    # ax.set_title("Population Density Zones", fontsize=14)
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

def generate_poi_map(poi_gdf, edges, extent, filename):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)
    
    poi_gdf = poi_gdf.to_crs(TARGET_CRS)
    categories = sorted(poi_gdf["category_level_0"].dropna().unique())
    cmap = plt.get_cmap("tab20", len(categories))
    
    for i, cat in enumerate(categories):
        subset = poi_gdf[poi_gdf["category_level_0"] == cat]
        subset.plot(ax=ax, color=cmap(i), markersize=5, label=cat, zorder=3)
    
    ax.legend(loc="lower right", fontsize=6, title="POIs Categories", ncol=2)
    # ax.set_title("POI Distribution", fontsize=14)
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

def generate_sensor_map(sensors_gdf, edges, extent, filename):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)
    
    sensors_gdf.to_crs(TARGET_CRS).plot(
        ax=ax, color="dodgerblue", edgecolor="white",
        linewidth=0.5, markersize=70, marker="^", zorder=5, label="AVI Sensor"
    )
    
    # ax.legend(loc="lower left", fontsize=8)
    # ax.set_title("Traffic Sensor Locations", fontsize=14)
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

In [ ]:
_, road_edges = get_infrastructure_by_extent()

In [ ]:
zoom_extent = get_plot_extent([census_gdf, gdf_pois, traffic_cams_df_with_dirs], padding=1000)

In [ ]:
# 4. Generate the three distinct images
generate_census_map(census_gdf, road_edges, zoom_extent, "./outputs/new_figs/map_census.pdf")
generate_poi_map(gdf_pois, road_edges, zoom_extent, "./outputs/new_figs/map_pois.pdf")
generate_sensor_map(traffic_cams_df_with_dirs, road_edges, zoom_extent, "./outputs/new_figs/map_sensors.pdf")

print("Maps generated successfully.")

In [ ]:
# All 95 sensors distribution plot

traffic_cam_95 = gpd.read_file("./preprocess/veh_counts_95_aug.geojson")

In [ ]:
generate_sensor_map(traffic_cam_95, road_edges, zoom_extent, "./outputs/new_figs/map_sensors_all_95.pdf")

In [ ]:
junc_aggr_cams = gpd.read_file("./data/prod/pre-process/traffic_cams_by_junc/gpd/traffic_cam_metadata_by_junc.shp")

generate_sensor_map(junc_aggr_cams, road_edges, zoom_extent, "./outputs/new_figs/map_sensors_by_junc.pdf")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pandas as pd
import geopandas as gpd

def generate_junction_residual_map(
    residuals_df: pd.DataFrame,
    junctions_gdf: gpd.GeoDataFrame,
    edges: gpd.GeoDataFrame,
    extent: list,
    filename: str,
    junction_id_col: str = "junction_id",  # column in junctions_gdf matching residuals_df index
):
    # --- Merge residuals into junctions ---
    junctions = junctions_gdf.copy().to_crs(TARGET_CRS)
    junctions = junctions.merge(
        residuals_df[["normalized_residual"]],
        left_on=junction_id_col,
        right_index=True,
        how="inner",
    )

    # --- Colormap: diverging, centered at 0 ---
    cmap = plt.get_cmap("RdYlGn")          # red = negative, green = positive
    norm = mcolors.TwoSlopeNorm(
        vmin=junctions["normalized_residual"].min(),
        vcenter=0.0,
        vmax=junctions["normalized_residual"].max(),
    )

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    plot_base_infrastructure(ax, edges, extent)

    junctions.plot(
        ax=ax,
        column="normalized_residual",
        cmap=cmap,
        norm=norm,
        markersize=40,
        marker="o",
        edgecolor="white",
        linewidth=0.4,
        zorder=5,
        legend=False,
    )

    # --- Colorbar ---
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.01)
    cbar.set_label("Normalized residual\n(inflow − outflow)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    ax.set_axis_off()
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    plt.close()

In [ ]:
junctions_gdf = gpd.read_file("./data/prod/pre-process/traffic_cams_by_junc/gpd/traffic_cam_metadata_by_junc.shp")
junctions_gdf

In [ ]:

residuals_df = pd.read_csv("./data/prod/pre-process/plate_hash/flow_residuals.csv", index_col="junctions")
# nodes, edges = get_infrastructure_by_extent()
# extent = get_plot_extent([nodes, junctions_gdf])

generate_junction_residual_map(
    residuals_df=residuals_df,
    junctions_gdf=junctions_gdf,
    edges=road_edges,
    extent=zoom_extent,
    filename="./outputs/new_figs/junction_residuals.pdf",
    junction_id_col="id",   # adjust to match your GeoDataFrame
)

## Traffic flow statistics

### Avg. travel times

In [ ]:
def plot_hourly_travel_time_profile(avg_travel_times_hourly: pd.DataFrame):
    df = avg_travel_times_hourly.copy()
    df['hour'] = df['time_bin'].dt.hour
    
    df['day_type'] = df['time_bin'].dt.dayofweek.map(
        lambda d: 'Weekend' if d >= 5 else 'Weekdays'
    )
    # df = df[df["day_type"] == "Weekdays"]
    # df = df[df["mean"] > 300]
    # df['day_type'] = "all"
    fig, ax = plt.subplots(figsize=(10, 5))

    styles = {
        'Weekdays': {'color': 'steelblue',  'marker': 'o'},
        'Weekend':           {'color': 'darkorange', 'marker': 's'},
        # "all":              {'color': 'steelblue',  'marker': 'o'}
    }

    for day_type, style in styles.items():
        stats = (
            df[df['day_type'] == day_type]
              .groupby('hour')['mean']
              .agg(['mean', 'std'])
        )
        ax.plot(stats.index, stats['mean'], marker=style['marker'],
                color=style['color'], label=day_type)
        # ax.fill_between(
        #     stats.index,
        #     stats['mean'] - stats['std'],
        #     stats['mean'] + stats['std'],
        #     alpha=0.3,
        #     color=style['color']
        # )

    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Average travel time (seconds)")
    # ax.set_title("Hourly Average Travel Time Profiles (Weekdays vs. Weekend)")
    hours = range(24)
    ax.set_xticks(hours)
    ax.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45, ha="right", fontsize=9)
    
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig("./outputs/new_figs/avg_travel_times.pdf", format="pdf", bbox_inches="tight")
    plt.show()

In [ ]:
avg_travel_times_hourly = pd.read_csv("./data/prod/pre-process/plate_hash/avg_travel_time_hour.csv", index_col=0, parse_dates=["time_bin"])

In [ ]:
avg_travel_times_hourly["time_bin"] = pd.to_datetime(avg_travel_times_hourly["time_bin"].values, utc=True).tz_convert("Europe/Rome")

In [ ]:
avg_travel_times_hourly_ = avg_travel_times_hourly[(avg_travel_times_hourly["count"] > 3) & (avg_travel_times_hourly["mean"] > 60)]

In [ ]:
len(avg_travel_times_hourly_) / len(avg_travel_times_hourly)

In [ ]:
plot_hourly_travel_time_profile(avg_travel_times_hourly_)

## POIs download example

This cell shows the code used to download and pre-process the POIs dataset saved in ./data/prod/pre-process/context/filtered_grouped_poi.csv

In [ ]:
import numpy as np

max_lat, min_lat = traffic_cams_df_with_dirs['LAT'].max(), traffic_cams_df_with_dirs['LAT'].min()
max_lon, min_lon = traffic_cams_df_with_dirs['LON'].max(), traffic_cams_df_with_dirs['LON'].min()

expand_m = 1000
expand_lat = expand_m / 111320.0
mean_lat = (max_lat + min_lat) / 2.0
expand_lon = expand_m / (111320.0 * np.cos(np.radians(mean_lat)))


print(f"Latitude range: {min_lat} to {max_lat}")
print(f"Longitude range: {min_lon} to {max_lon}")

In [ ]:
from pyproj import Geod

bbox = (
    min_lon - expand_lon,
    min_lat - expand_lat,
    max_lon + expand_lon,
    max_lat + expand_lat
)

print(bbox)

In [ ]:
! overturemaps download --bbox=11.81755880708736,45.357261888250086,11.94228119291264,45.45622111174991 -f geojson --type=place -o poi.geojson

In [ ]:
import json
import pandas as pd

with open("./poi.geojson", "r") as f:
    geojson_data = json.load(f)
    
records = []
features = geojson_data["features"]
for feat in features:
    props = feat["properties"]
    geom = feat["geometry"]
    record = {
        "id": props.get("id"),
        "name": props.get("names", {}).get("primary", "-"),
        "version": props.get("version"),
        "confidence": props.get("confidence"),
        "primary_category": props.get("categories", {}).get("primary"),
        "longitude": geom["coordinates"][0] if geom and "coordinates" in geom else None,
        "latitude": geom["coordinates"][1] if geom and "coordinates" in geom else None
    }
    records.append(record)


df = pd.DataFrame(records)

In [ ]:
def filter_by_confidence(df, threshold):
    return df[df["confidence"] >= threshold]

threshold = 0.7
filtered_df = filter_by_confidence(df, threshold)
print(f"\nEntries with confidence >= {threshold}:", len(filtered_df))

In [ ]:
# Categories Taxonomy
import ast

def transform_taxonomy(taxonomy_str):
    items = taxonomy_str.strip("[]").split(",")
    items = [item.strip() for item in items if item.strip()]
    return "[" + ",".join(f'"{item}"' for item in items) + "]"


def build_category_mapping(taxonomy_df, level):
    mapping = {}
    for _, row in taxonomy_df.iterrows():
        cat = row["Category code"]
        path = ast.literal_eval(row["Overture Taxonomy"])
        if level < len(path):
            mapping[cat] = path[level]
        else:
            mapping[cat] = path[-1]
    return mapping


def map_poi_categories(df, taxonomy_df, level):
    mapping = build_category_mapping(taxonomy_df, level)
    df = df.copy()
    df["category_level_" + str(level)] = df["primary_category"].map(mapping)
    return df


## Load overture categories
taxonomy = pd.read_csv("./data/prod/overture_categories.csv", sep=";")
taxonomy.columns = [c.strip() for c in taxonomy.columns]
taxonomy = taxonomy.map(lambda x: x.strip() if isinstance(x, str) else x)
taxonomy["Overture Taxonomy"] = taxonomy["Overture Taxonomy"].apply(transform_taxonomy)

level = 0
df_grouped_cat = map_poi_categories(filtered_df, taxonomy, level)

num_distinct_categories = df_grouped_cat["category_level_0"].nunique()
print("Number of distinct categories:", num_distinct_categories)

In [ ]:
df_grouped_cat.head()

## Spatial Aggregation Example

This cell shows the code used to perform spatial aggregation and obtain data saved in ./data/prod/pre-process/traffic_cams_by_junc/

In [ ]:
grouped_df, agg_list, mapping = merge_data_by_junc(
    data = aggregated_timeseries, 
    metadata = traffic_cams_df_with_dirs,
    use_direction=True,
    offset=0,
    eps=50
)

## Corr analysis

In [ ]:
# corr_matrix = aggregated_timeseries.corr(method="pearson")

# n = corr_matrix.shape[0]
# upper_idx = np.triu_indices(n, k=1)          # k=1 skips the diagonal
# pairwise_corrs = corr_matrix.values[upper_idx]

# print(f"\nNumber of node pairs: {len(pairwise_corrs)}")
# print(f"Pearson correlation  –  mean: {pairwise_corrs.mean():.4f} "
#       f"| std: {pairwise_corrs.std():.4f} "
#       f"| min: {pairwise_corrs.min():.4f} "
#       f"| max: {pairwise_corrs.max():.4f}")

# all_values = aggregated_timeseries.values.flatten()
# all_values = all_values[~np.isnan(all_values)]   # remove NaNs

# print(f"\nTraffic values  –  mean: {all_values.mean():.2f} "
#       f"| std: {all_values.std():.2f} "
#       f"| min: {all_values.min():.2f} "
#       f"| max: {all_values.max():.2f}")

# fig, ax = plt.subplots(figsize=(6, 4))

# # Compute true probabilities: each bar = fraction of total pairs
# counts, bin_edges = np.histogram(pairwise_corrs, bins=20, range=(-0.2, 1.0))
# probs = counts / counts.sum()  # now each bar is a true probability, and they sum to 1

# ax.bar(
#     x=bin_edges[:-1],           # left edge of each bin
#     height=probs,
#     width=np.diff(bin_edges),   # bin width
#     align="edge",
#     color="steelblue",
#     edgecolor="white"
# )

# ax.set_xlabel("Pearson correlation")
# ax.set_ylabel("Probability")
# ax.set_title("Distribution of inter-node correlations")
# ax.set_xlim(-0.2, 1.0)
# ax.set_ylim(0, None)

# plt.tight_layout()
# plt.show()